In [1]:
!apt-get install -y rabbitmq-server > /dev/null 2>&1
!service rabbitmq-server start
!pip install -q pika
!mkdir -p stream
print("RabbitMQ Server is running!")

[ OK ]
RabbitMQ Server is running!


In [2]:
%%writefile stream/dispatcher.py
import pika, json, time, random, uuid, os

EXCHANGE_NAME = "rapido-dispatch"
RABBITMQ_URL = os.environ.get("RABBITMQ_URL", "amqp://localhost")

CITIES = ["bangalore", "chennai"]
ZONES = {
    "bangalore": ["koramangala", "indiranagar", "whitefield", "jayanagar"],
    "chennai": ["tnagar", "velachery", "adyar"]
}
TYPES = ["ride", "food"]

connection = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
channel = connection.channel()
channel.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

msg_count = 0
print("[*] Dispatcher started. Publishing every 2s...\n")

try:
    while True:
        msg_count += 1
        city = random.choice(CITIES)
        zone = random.choice(ZONES[city])
        order_type = random.choice(TYPES)
        order_id = f"ORD-{msg_count:03d}"
        ts = time.strftime("%H:%M:%S")

        # Every 10th message is intentionally corrupted (missing 'zone') for D3
        is_bad = (msg_count % 10 == 0)

        if is_bad:
            routing_key = f"{city}.unknown.{order_type}"
            body = {"id": order_id, "city": city, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #{msg_count} (missing zone) -> Key: '{routing_key}'")
        else:
            routing_key = f"{city}.{zone}.{order_type}"
            body = {"id": order_id, "city": city, "zone": zone, "type": order_type, "ts": ts}
            print(f"[DISPATCHER] Sent #{msg_count}: Key='{routing_key}' | ID={order_id}")

        channel.basic_publish(
            exchange=EXCHANGE_NAME,
            routing_key=routing_key,
            body=json.dumps(body)
        )
        time.sleep(2)
except KeyboardInterrupt:
    print("\nDispatcher stopped.")
finally:
    connection.close()

Writing stream/dispatcher.py


In [5]:
import threading, time, json, random, pika, os

EXCHANGE_NAME = "rapido-dispatch"
RABBITMQ_URL = "amqp://localhost"
stop_event = threading.Event()

# ── 1. DRIVER THREAD FUNCTION (matches driver.py) ──────────────
def run_driver(name, pattern):
    conn = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
    ch = conn.channel()
    ch.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

    q = ch.queue_declare(queue='', exclusive=True, auto_delete=True).method.queue
    ch.queue_bind(exchange=EXCHANGE_NAME, queue=q, routing_key=pattern)
    ch.basic_qos(prefetch_count=1)
    print(f"[{name}] Bound to '{pattern}'")

    def on_msg(ch, method, props, body):
        data = json.loads(body.decode())
        oid = data.get("id")
        rk = method.routing_key

        # D3 ACK / NACK validation
        if "zone" not in data or data.get("zone") is None:
            print(f"   └── [{name}] ❌ [NACK] Rejected {oid} (missing zone) | key='{rk}'")
            ch.basic_nack(delivery_tag=method.delivery_tag, requeue=False)
        else:
            print(f"   └── [{name}] ✅ [ACK] Accepted {oid} | {data['city']}.{data['zone']}.{data['type']}")
            ch.basic_ack(delivery_tag=method.delivery_tag)

    ch.basic_consume(queue=q, on_message_callback=on_msg, auto_ack=False)

    while not stop_event.is_set():
        conn.process_data_events(time_limit=0.2)
    conn.close()

# ── 2. DISPATCHER THREAD FUNCTION (matches dispatcher.py) ─────
def run_dispatcher():
    conn = pika.BlockingConnection(pika.URLParameters(RABBITMQ_URL))
    ch = conn.channel()
    ch.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

    CITIES = ["bangalore", "chennai"]
    ZONES = {
        "bangalore": ["koramangala", "indiranagar", "whitefield"],
        "chennai": ["tnagar"]
    }
    TYPES = ["ride", "food"]
    count = 0

    while not stop_event.is_set():
        count += 1
        city = random.choice(CITIES)
        zone = random.choice(ZONES[city])
        order_type = random.choice(TYPES)
        oid = f"ORD-{count:03d}"
        ts = time.strftime("%H:%M:%S")

        # Every 10th message is intentionally corrupted for D3
        is_bad = (count % 10 == 0)

        if is_bad:
            rk = f"{city}.unknown.{order_type}"
            body = {"id": oid, "city": city, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #{count} (missing zone) -> key='{rk}'")
        else:
            rk = f"{city}.{zone}.{order_type}"
            body = {"id": oid, "city": city, "zone": zone, "type": order_type, "ts": ts}
            print(f"\n[DISPATCHER] Sent #{count}: key='{rk}' | ID={oid}")

        ch.basic_publish(exchange=EXCHANGE_NAME, routing_key=rk, body=json.dumps(body))

        # Sleep in short slices so stop_event is responsive
        for _ in range(10):
            if stop_event.is_set(): break
            time.sleep(0.2)
    conn.close()

# ── 3. LAUNCH ALL THREADS CONCURRENTLY ─────────────────────────
stop_event.clear()
drivers = [
    ("Ravi",    "bangalore.koramangala.*"),
    ("Meena",   "bangalore.*.food"),
    ("Control", "bangalore.#")
]

threads = [threading.Thread(target=run_driver, args=(name, pat), daemon=True) for name, pat in drivers]
threads.append(threading.Thread(target=run_dispatcher, daemon=True))

for t in threads: t.start()
print("All threads running! Waiting for dispatches...\n")

try:
    time.sleep(45)  # Run for 45 seconds (or press Stop ■ anytime)
except KeyboardInterrupt:
    pass
finally:
    stop_event.set()
    time.sleep(1)
    print("\n[✓] Stopped cleanly.")

All threads running! Waiting for dispatches...


[DISPATCHER] Sent #1: key='bangalore.koramangala.food' | ID=ORD-001
[Control] Bound to 'bangalore.#'
[Meena] Bound to 'bangalore.*.food'
[Ravi] Bound to 'bangalore.koramangala.*'

[DISPATCHER] Sent #2: key='bangalore.whitefield.ride' | ID=ORD-002
   └── [Control] ✅ [ACK] Accepted ORD-002 | bangalore.whitefield.ride

[DISPATCHER] Sent #3: key='chennai.tnagar.food' | ID=ORD-003

[DISPATCHER] Sent #4: key='chennai.tnagar.ride' | ID=ORD-004

[DISPATCHER] Sent #5: key='chennai.tnagar.ride' | ID=ORD-005

[DISPATCHER] Sent #6: key='bangalore.koramangala.food' | ID=ORD-006
   └── [Ravi] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food
   └── [Control] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food
   └── [Meena] ✅ [ACK] Accepted ORD-006 | bangalore.koramangala.food

[DISPATCHER] Sent #7: key='chennai.tnagar.ride' | ID=ORD-007

[DISPATCHER] Sent #8: key='bangalore.whitefield.food' | ID=ORD-008
   └── [Control] ✅ [ACK] Accepted ORD-008

In [4]:
import subprocess, time, os

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Launch 3 drivers in the background
d_ravi = subprocess.Popen(["python", "-u", "stream/driver.py", "Ravi", "bangalore.koramangala.*"], env=env)
d_meena = subprocess.Popen(["python", "-u", "stream/driver.py", "Meena", "bangalore.*.food"], env=env)
d_control = subprocess.Popen(["python", "-u", "stream/driver.py", "Control", "bangalore.#"], env=env)

time.sleep(2)  # Wait for drivers to bind

# Run dispatcher
try:
    !python -u stream/dispatcher.py
except KeyboardInterrupt:
    pass
finally:
    d_ravi.terminate()
    d_meena.terminate()
    d_control.terminate()
    print("All processes stopped cleanly.")

[*] Dispatcher started. Publishing every 2s...

[DISPATCHER] Sent #1: Key='chennai.tnagar.ride' | ID=ORD-001
[DISPATCHER] Sent #2: Key='bangalore.indiranagar.ride' | ID=ORD-002
[DISPATCHER] Sent #3: Key='bangalore.koramangala.food' | ID=ORD-003
[DISPATCHER] Sent #4: Key='bangalore.whitefield.food' | ID=ORD-004
[DISPATCHER] Sent #5: Key='bangalore.jayanagar.ride' | ID=ORD-005
[DISPATCHER] Sent #6: Key='bangalore.koramangala.food' | ID=ORD-006
[DISPATCHER] Sent #7: Key='chennai.adyar.ride' | ID=ORD-007
[DISPATCHER] Sent #8: Key='bangalore.jayanagar.ride' | ID=ORD-008
[DISPATCHER] Sent #9: Key='chennai.velachery.ride' | ID=ORD-009

[DISPATCHER] ⚠️ DISPATCHING BAD MESSAGE #10 (missing zone) -> Key: 'chennai.unknown.food'
[DISPATCHER] Sent #11: Key='bangalore.whitefield.food' | ID=ORD-011
[DISPATCHER] Sent #12: Key='chennai.adyar.food' | ID=ORD-012
[DISPATCHER] Sent #13: Key='chennai.velachery.food' | ID=ORD-013
[DISPATCHER] Sent #14: Key='chennai.adyar.ride' | ID=ORD-014
[DISPATCHER] Sent 

In [7]:
!pip install snowflake-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 30.6 MB/s  0:00:00


In [9]:
!mkdir -p stream

In [12]:
%%writefile stream/stream_to_lakehouse.py
"""
stream/stream_to_lakehouse.py - BONUS (+10 Marks)
Connects Part D (RabbitMQ Event Stream) to Part B (FreshCart Snowflake Lakehouse).
"""

import gzip
import json
import os
from pathlib import Path
import time
from datetime import datetime, timezone
import pika

RABBITMQ_URL = os.environ.get("RABBITMQ_URL", "amqp://localhost")
STREAM_QUEUE = "freshcart-stream-landing"
EXCHANGE = "rapido-dispatch"
LANDING_DIR = os.environ.get("LANDING_DIR", "/content/freshcart_landing")


def write_landing(source, records, run_id):
    """Saves source records in a compressed JSON Lines file and returns its path."""
    folder = Path(LANDING_DIR) / source
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{source}__{run_id}.json.gz"
    now = datetime.now(timezone.utc).isoformat()

    with gzip.open(path, "wt", encoding="utf-8") as f:
        for rec in records:
            envelope = {
                "source": source,
                "run_id": run_id,
                "extracted_at": now,
                "data": rec
            }
            f.write(json.dumps(envelope, ensure_ascii=False) + "\n")

    print(f"   [Landing] {source}: {len(records)} records -> {path}")
    return path


def collect_orders_from_stream(duration_seconds=20):
    """Listens to RabbitMQ and collects streamed orders for duration_seconds."""
    print(f"[*] Connecting to RabbitMQ to collect streaming orders for {duration_seconds}s...")
    params = pika.URLParameters(RABBITMQ_URL)
    conn = pika.BlockingConnection(params)
    ch = conn.channel()

    ch.queue_declare(queue=STREAM_QUEUE, durable=True)
    ch.queue_bind(exchange=EXCHANGE, queue=STREAM_QUEUE, routing_key="bangalore.*.*")

    collected_orders = []
    start_time = time.time()
    print(f"[*] Consumer listening on '{STREAM_QUEUE}' (binding: 'bangalore.*.*')...\n")

    while (time.time() - start_time) < duration_seconds:
        method, props, body = ch.basic_get(queue=STREAM_QUEUE, auto_ack=False)
        if method:
            order = json.loads(body.decode())
            freshcart_order = {
                "order_id": order.get("id"),
                "cust_email": "arjun.k@yahoo.in",
                "cust_name": "Arjun K",
                "order_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
                "amount": "450.00",
                "currency": "INR",
                "pincode": "560034",
                "updated_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
            collected_orders.append(freshcart_order)
            ch.basic_ack(method.delivery_tag)
            print(f"   [+] Buffered streamed order {freshcart_order['order_id']} ({len(collected_orders)} in batch)")
        time.sleep(0.5)

    conn.close()
    print(f"\n[✓] {duration_seconds}s window closed. Collected {len(collected_orders)} orders from RabbitMQ.")
    return collected_orders


def load_file(cursor, local_path, stage_dir, run_id):
    """Uploads raw file to Snowflake stage or directly into bronze.orders_raw."""
    try:
        cursor.execute("USE WAREHOUSE COMPUTE_WH")
        cursor.execute("USE DATABASE FRESHCART")
        cursor.execute("USE SCHEMA BRONZE")
    except Exception as e:
        print(f"   [!] Context notice: {e}")

    uri = "file://" + Path(local_path).resolve().as_posix()
    print(f"   [Stage] Uploading '{uri}' to stage @FRESHCART.BRONZE.LANDING/{stage_dir}/...")

    try:
        cursor.execute(f"PUT '{uri}' @FRESHCART.BRONZE.LANDING/{stage_dir}/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE")
        cursor.execute(f"""
            COPY INTO FRESHCART.BRONZE.orders_raw (source, raw, run_id, extracted_at, src_file, src_row)
            FROM (
              SELECT $1:source::STRING, $1:data, $1:run_id::STRING,
                     $1:extracted_at::TIMESTAMP_TZ, METADATA$FILENAME, METADATA$FILE_ROW_NUMBER
              FROM @FRESHCART.BRONZE.LANDING/{stage_dir}/
            )
            FILE_FORMAT = (FORMAT_NAME = 'FRESHCART.BRONZE.ff_ndjson')
            ON_ERROR = 'ABORT_STATEMENT'
        """)
        print("   [✓] COPY INTO bronze.orders_raw via stage succeeded.")
    except Exception as err:
        print(f"   [Notice] Stage PUT skipped ({err}). Using direct landing ingest...")
        row_count = 0
        with gzip.open(local_path, "rt", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    cursor.execute(
                        """
                        INSERT INTO FRESHCART.BRONZE.orders_raw (source, raw, run_id, extracted_at, src_file, src_row)
                        SELECT %s, PARSE_JSON(%s), %s, %s, %s, %s
                        """,
                        (
                            rec.get("source", "web"),
                            json.dumps(rec.get("data")),
                            rec.get("run_id", run_id),
                            rec.get("extracted_at"),
                            Path(local_path).name,
                            row_count + 1
                        )
                    )
                    row_count += 1
        print(f"   [✓] Direct bronze ingestion complete: {row_count} records inserted into bronze.orders_raw.")


def run_lakehouse_ingestion(snowflake_conn, collected_orders):
    """Loads collected stream batch into Snowflake bronze and merges to gold."""
    if not collected_orders:
        print("[!] No orders collected in this window. Exiting ingestion.")
        return

    run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    cursor = snowflake_conn.cursor()

    landing_file = write_landing("web", collected_orders, run_id)

    print("\n[Stage 1] Loading raw file into bronze.orders_raw via load_file()...")
    load_file(cursor, landing_file, "web", run_id)

    print("\n[Stage 2] Running ELT Transformations (Silver Clean -> Dedup -> Gold)...")
    cursor.execute("USE DATABASE FRESHCART")

    cursor.execute("""
        CREATE OR REPLACE TRANSIENT TABLE silver.orders_checked AS
        WITH unified AS (
          SELECT ingest_seq, source,
                 raw:order_id::STRING   AS order_id,
                 raw:cust_email::STRING AS cust_email,
                 raw:cust_name::STRING  AS cust_name,
                 raw:order_date::STRING AS order_date,
                 raw:amount::STRING     AS amount,
                 raw:currency::STRING   AS currency,
                 raw:pincode::STRING    AS pincode,
                 raw:updated_at::STRING AS updated_at
          FROM bronze.orders_raw WHERE source = 'web'
          UNION ALL
          SELECT ingest_seq, source,
                 raw:orderId::STRING, raw:customer.email::STRING, raw:customer.name::STRING,
                 raw:orderDate::STRING, raw:total.value::STRING, raw:total.currency::STRING,
                 raw:deliveryPin::STRING, raw:updatedAt::STRING
          FROM bronze.orders_raw WHERE source = 'mobile'
        ),
        typed AS (
          SELECT
            ingest_seq, source,
            TRIM(order_id)                                    AS order_id,
            LOWER(TRIM(cust_email))                           AS email,
            INITCAP(RTRIM(TRIM(cust_name), '.'))              AS name,
            COALESCE(
              TRY_TO_DATE(order_date, 'YYYY-MM-DD'),
              TRY_TO_DATE(order_date, 'DD/MM/YYYY'),
              TRY_TO_DATE(order_date, 'MON DD, YYYY')
            )                                                 AS order_date,
            TRY_TO_DECIMAL(REGEXP_REPLACE(amount, '[^0-9.-]', ''), 12, 2) AS amount,
            UPPER(TRIM(currency))                             AS currency,
            TRIM(pincode)                                     AS pincode,
            TRY_TO_TIMESTAMP_NTZ(updated_at)                  AS updated_at
          FROM unified
        )
        SELECT *,
          ARRAY_CONSTRUCT_COMPACT(
            IFF(order_id IS NULL, 'missing_order_id', NULL),
            IFF(email IS NULL OR NOT REGEXP_LIKE(email, '^[^@\\\\s]+@[^@\\\\s]+\\\\.[a-z]{2,}$'), 'bad_email', NULL),
            IFF(pincode IS NULL OR NOT REGEXP_LIKE(pincode, '^[1-9][0-9]{5}$'), 'bad_pincode', NULL),
            IFF(amount IS NULL OR amount <= 0, 'bad_or_non_positive_amount', NULL),
            IFF(currency IS NULL OR currency NOT IN ('INR', 'USD', 'EUR', 'GBP'), 'unsupported_currency', NULL)
          ) AS reasons
        FROM typed
    """)

    cursor.execute("""
        CREATE OR REPLACE TABLE silver.orders_clean AS
        SELECT * EXCLUDE reasons FROM silver.orders_checked WHERE ARRAY_SIZE(reasons) = 0
    """)

    cursor.execute("""
        CREATE OR REPLACE TABLE silver.orders_dedup AS
        SELECT * FROM silver.orders_clean
        QUALIFY ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC, ingest_seq DESC) = 1
    """)

    cursor.execute("""
        MERGE INTO gold.fct_orders t
        USING (
          SELECT
            d.order_id, d.order_date, d.email, d.name, d.pincode,
            d.amount, d.currency,
            ROUND(d.amount * COALESCE(f.rate_to_inr, 1.0), 2) AS amount_inr,
            FLOOR(ROUND(d.amount * COALESCE(f.rate_to_inr, 1.0), 2) / 100) * 2 AS loyalty_points,
            HASH(d.order_id, d.order_date, d.email, d.amount, d.currency) AS row_hash
          FROM silver.orders_dedup d
          LEFT JOIN silver.fx_rates f ON d.currency = f.currency
        ) s
        ON t.order_id = s.order_id
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
          amount = s.amount, amount_inr = s.amount_inr, loyalty_points = s.loyalty_points,
          row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
        WHEN NOT MATCHED THEN INSERT
          (order_id, order_date, email, name, pincode, amount, currency, amount_inr, loyalty_points, row_hash, published_at)
        VALUES
          (s.order_id, s.order_date, s.email, s.name, s.pincode, s.amount, s.currency, s.amount_inr, s.loyalty_points, s.row_hash, CURRENT_TIMESTAMP())
    """)
    print("[Stage 3] Ingestion and Gold MERGE complete.")


def verify_idempotency(snowflake_conn):
    """Proves re-running the ELT batch produces 0 duplicate rows and 0 double-counting."""
    cursor = snowflake_conn.cursor()
    cursor.execute("USE DATABASE FRESHCART")

    print("\n" + "="*65)
    print("IDEMPOTENCY PROOF: Checking gold totals after first run...")
    cursor.execute("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
    first_orders, first_rev, first_pts = cursor.fetchone()
    print(f"   [Run 1] Total Orders: {first_orders} | Revenue: ₹{first_rev:,.2f} | Points: {first_pts}")

    print("\nRe-executing MERGE pipeline with the same stream batch (Second Run)...")
    cursor.execute("""
        MERGE INTO gold.fct_orders t
        USING (
          SELECT
            d.order_id, d.order_date, d.email, d.name, d.pincode,
            d.amount, d.currency,
            ROUND(d.amount * COALESCE(f.rate_to_inr, 1.0), 2) AS amount_inr,
            FLOOR(ROUND(d.amount * COALESCE(f.rate_to_inr, 1.0), 2) / 100) * 2 AS loyalty_points,
            HASH(d.order_id, d.order_date, d.email, d.amount, d.currency) AS row_hash
          FROM silver.orders_dedup d
          LEFT JOIN silver.fx_rates f ON d.currency = f.currency
        ) s
        ON t.order_id = s.order_id
        WHEN MATCHED AND t.row_hash <> s.row_hash THEN UPDATE SET
          amount = s.amount, amount_inr = s.amount_inr, loyalty_points = s.loyalty_points,
          row_hash = s.row_hash, published_at = CURRENT_TIMESTAMP()
        WHEN NOT MATCHED THEN INSERT
          (order_id, order_date, email, name, pincode, amount, currency, amount_inr, loyalty_points, row_hash, published_at)
        VALUES
          (s.order_id, s.order_date, s.email, s.name, s.pincode, s.amount, s.currency, s.amount_inr, s.loyalty_points, s.row_hash, CURRENT_TIMESTAMP())
    """)

    cursor.execute("SELECT COUNT(*), SUM(amount_inr), SUM(loyalty_points) FROM gold.fct_orders")
    second_orders, second_rev, second_pts = cursor.fetchone()
    print(f"   [Run 2] Total Orders: {second_orders} | Revenue: ₹{second_rev:,.2f} | Points: {second_pts}")
    print("="*65)

    assert first_orders == second_orders, "Idempotency failed: Order count changed!"
    assert first_rev == second_rev, "Idempotency failed: Revenue doubled or changed!"
    assert first_pts == second_pts, "Idempotency failed: Loyalty points changed!"
    print("\n✅ IDEMPOTENCY VERIFIED: Re-running batch produced 0 duplicate rows, 0 double counting!")

Overwriting stream/stream_to_lakehouse.py


In [13]:
import subprocess, time, os, snowflake.connector
from getpass import getpass

# 1. Ask for your Snowflake password
SNOWFLAKE_PASSWORD = getpass("Snowflake password: ")

# 2. Start the dispatcher in the background (starts streaming orders into RabbitMQ)
print("\n[Step 1] Starting dispatcher in background...")
dispatcher_proc = subprocess.Popen(["python", "-u", "stream/dispatcher.py"])
time.sleep(2)

# 3. Connect to Snowflake
print("[Step 2] Connecting to Snowflake...")
conn = snowflake.connector.connect(
    account="QJWRQZA-BS78896",
    user="NIKVED1705",
    password=SNOWFLAKE_PASSWORD,
    role="ACCOUNTADMIN",
    warehouse="COMPUTE_WH",
    database="FRESHCART_DB",
    schema="PUBLIC"
)
print("Connected to Snowflake!")

# 4. Run the Stream-to-Lakehouse bridge!
import stream.stream_to_lakehouse as bridge

# Listen to RabbitMQ for 30 seconds, collect orders, load to Snowflake, and verify idempotency:
orders = bridge.collect_orders_from_stream(duration_seconds=30)
bridge.run_lakehouse_ingestion(conn, orders)
bridge.verify_idempotency(conn)

# 5. Stop background dispatcher
dispatcher_proc.terminate()
print("\n[✓] Stream-to-Lakehouse pipeline verified live!")

Snowflake password: ··········

[Step 1] Starting dispatcher in background...


/usr/local/lib/python3.13/dist-packages/snowflake/connector/connection/_connection.py:137: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


[Step 2] Connecting to Snowflake...
Connected to Snowflake!
[*] Connecting to RabbitMQ to collect streaming orders for 30s...
[*] Consumer listening on 'freshcart-stream-landing' (binding: 'bangalore.*.*')...

   [+] Buffered streamed order ORD-274 (1 in batch)
   [+] Buffered streamed order ORD-018 (2 in batch)
   [+] Buffered streamed order ORD-020 (3 in batch)
   [+] Buffered streamed order ORD-277 (4 in batch)
   [+] Buffered streamed order ORD-022 (5 in batch)
   [+] Buffered streamed order ORD-023 (6 in batch)
   [+] Buffered streamed order ORD-025 (7 in batch)
   [+] Buffered streamed order ORD-282 (8 in batch)
   [+] Buffered streamed order ORD-026 (9 in batch)
   [+] Buffered streamed order ORD-027 (10 in batch)
   [+] Buffered streamed order ORD-029 (11 in batch)
   [+] Buffered streamed order ORD-286 (12 in batch)
   [+] Buffered streamed order ORD-287 (13 in batch)
   [+] Buffered streamed order ORD-031 (14 in batch)
   [+] Buffered streamed order ORD-288 (15 in batch)
   [

ProgrammingError: 090105 (22000): Cannot perform STAGE PUT. This session does not have a current database. Call 'USE DATABASE', or use a qualified name. (request_id=08ab869d-7c32-4ca4-9953-63604fa9cd25, sfqid=01c86d5f-0003-1527-000f-63220002953a)